# Homework 9: least squares, comparing fits, and outliers

Covers Lecture 22 (a least-squares objective, several parameters, `res.x`, and `curve_fit` as a check), Lecture 23 (`rmse`, comparing model forms, and a known point), and Lecture 24 (outliers, mean absolute error with `minimize`, and selecting rows to refit).

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning.
Reloading or disconnecting can revert the notebook to the instructor copy.

# Problem 0 (-10 pts if blank): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work. Critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

# Foam data

A lab compresses a block of polyurethane foam and records the force at 17 displacements, from 0 to 8 mm.
The first reading is at zero displacement and zero force.

*Data: constructed for this assignment from an exponential curve, with measurement noise added.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

displacement_mm = np.array([0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8])   # mm
force_kn = np.array([0, -0.001, 0.012, 0.022, 0.033, 0.068, 0.075, 0.116, 0.139,
                     0.203, 0.272, 0.358, 0.454, 0.628, 0.826, 1.097, 1.454])                       # kN

print(len(displacement_mm), 'readings')

17 readings


`displacement_mm` is the displacement of each reading, in mm, and `force_kn` is the force at that displacement, in kN.

# Problem 1 (25 pts): a least-squares objective and `minimize`

(a) Write a function `exp_model(x, a, b)` that returns $a\,(e^{bx} - 1)$.

- Write a docstring that gives the unit of `x`, of each parameter, and of the result.
- Print `exp_model(4, 0.02, 0.5)`.

> Use $x$ in mm, $a$ in kN, and $b$ in 1/mm.

In [2]:
# your code here

(b) Write an objective function `sse_foam(params)`.

- Unpack `a, b = params` on the first line.
- Build the residuals from `displacement_mm` and `force_kn`.
- Return the sum of the squared residuals.
- Print `sse_foam([0.02, 0.5])`.
- Print `sse_foam([0.005, 1])`.

> `sse_foam` reads the two arrays from the notebook, so its only argument is `params`.

In [3]:
# your code here

(c) Find the parameters with `minimize`.

- Import `optimize` from `scipy`.
- Call `optimize.minimize(sse_foam, [0.05, 0.5])` and store the result as `res`.
- Unpack `res.x` as `a_min, b_min`.
- Print `a_min` with `kN`, rounded to four decimals.
- Print `b_min` with `1/mm`, rounded to three decimals.
- Print `res.fun`.
- Print `res.success`.

> `res.x` follows the order of the unpack line in `sse_foam`.

In [4]:
# your code here

(d) Fit the same model with `curve_fit`.

- Call `optimize.curve_fit(exp_model, displacement_mm, force_kn)` and unpack the result as `popt, pcov`.
- Print `popt[0]` with `kN`, rounded to four decimals.
- Print `popt[1]` with `1/mm`, rounded to three decimals.
- Plot the readings as dots.
- Plot the fitted curve as a line.
- Label both axes with units.

> Use `np.linspace(0, 8, 100)` for the smooth curve.

In [5]:
# your code here

(e) Compare the results of parts (c) and (d).

- In one or two sentences, explain why both calls give the same `a` and `b`.

*your answer here*

# Problem 2 (30 pts): `rmse` and comparing model forms

(a) Write a function `rmse(y, y_fit)` that returns the root-mean-square error.

- Write a docstring that says the result is in the units of `y`.
- Print `rmse(np.array([1, 2, 3]), np.array([1, 2, 5]))`.

In [6]:
# your code here

(b) Fit three forms to the readings and score each one.

- Import `stats` from `scipy`.
- Build `force_line` from a `stats.linregress` fit of `force_kn` against `displacement_mm`.
- Build `force_exp` from a `curve_fit` of `exp_model`, and store the fitted values as `popt_exp`.
- Write `quadratic(x, a, b, c)` that returns $a x^2 + b x + c$.
- Build `force_quad` from a `curve_fit` of `quadratic`, and store the fitted values as `popt_quad`.
- Print the RMSE of each form with `kN`, rounded to three decimals.

In [7]:
# your code here

(c) Plot the residuals of the three fits against `displacement_mm`.

- Plot each residual set as dots with a label.
- Draw a black line at zero with `ax.axhline`.
- Label both axes with units.
- Add a legend.

In [8]:
# your code here

(d) Check each form against the known point at zero displacement.

- Print the line's force at zero displacement, with `kN`, rounded to three decimals.
- Print the quadratic's force at zero displacement, in the same format.
- Print the exponential's force at zero displacement, in the same format.

In [9]:
# your code here

(e) Choose a form to report for this foam.

- Say which of the three forms you would report.
- Say why, in one or two sentences, using the RMSE, the residual plot, and the force at zero displacement.

*your answer here*

# Problem 3 (20 pts): `minimize` with a mean absolute error objective

An aluminum rod 500 mm long is heated in a furnace, and a dial gauge records its change in length, in µm, at eight temperatures from 20 °C to 300 °C.

The reading at 260 °C was entered as 27580 µm, and the gauge showed 2758 µm.

*Data: constructed for this assignment, with gauge noise added.*

In [10]:
temp_c = np.array([20, 60, 100, 140, 180, 220, 260, 300])                       # °C
dl_clean = np.array([0, 461, 911, 1379, 1841, 2297, 2758, 3221])                # µm
dl_typo = np.array([0, 461, 911, 1379, 1841, 2297, 27580, 3221])                # µm, extra digit at 260 °C

(a) Fit a line to each array.

- Fit `dl_clean` against `temp_c` with `stats.linregress`, and store the result as `fit_clean`.
- Fit `dl_typo` against `temp_c` with `stats.linregress`, and store the result as `fit_typo`.
- Print the slope of `fit_clean` with `µm/°C`, rounded to two decimals.
- Print the slope of `fit_typo` with `µm/°C`, rounded to two decimals.

In [11]:
# your code here

(b) Find the share of the sum of squared residuals from the reading at 260 °C.

- Compute the residuals of the typo fit, `dl_typo - (fit_typo.slope * temp_c + fit_typo.intercept)`.
- Print the percentage of their summed squares that comes from `dl_typo[6]`, rounded to the nearest whole number.

In [12]:
# your code here

(c) Fit the typo readings by minimizing the mean absolute error.

- Write an objective function `mae_rod(params)` that unpacks `m, b = params`.
- Build the residuals from `temp_c` and `dl_typo`.
- Return `np.mean(np.abs(residuals))`.
- Call `optimize.minimize(mae_rod, [1, 1], method='Nelder-Mead')` and store the result as `res_mae`.
- Print the slope, `res_mae.x[0]`, with `µm/°C`, rounded to two decimals.

> Copy `method='Nelder-Mead'` into the call as it is.

In [13]:
# your code here

(d) Compare the two slopes from parts (a) and (c).

- In one or two sentences, explain why the one wrong reading moves the least-squares slope much further than the minimum-MAE slope.

*your answer here*

# Problem 4 (25 pts): selecting rows with a mask and refitting

The table holds 915 tensile tests on low-alloy steels at temperatures from 27 °C to 650 °C.

You'll fit the tensile strength against the 0.2% proof stress, the stress at which a steel has deformed permanently by 0.2%.

*Data: NIMS [MatNavi](https://mits.nims.go.jp/en/), tensile tests on low-alloy steels.*

In [14]:
steels = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/steels.csv')

proof_col = ' 0.2% Proof Stress (MPa)'        # note the leading space
tensile_col = ' Tensile Strength (MPa)'       # note the leading space

proof = steels[proof_col]                      # MPa
tensile = steels[tensile_col]                  # MPa
print(len(steels), 'tests')

915 tests


(a) Fit a line to all of the tests.

- Call `stats.linregress(proof, tensile)` and store the result as `fit_all`.
- Print the slope, rounded to three decimals.
- Print R², rounded to two decimals.

In [15]:
# your code here

(b) Find the test with a tensile strength far above the fitted line.

- Compute `resid_all`, the tensile strength minus the fitted line.
- Print the alloy code, temperature, proof stress, tensile strength, and elongation of every test with a residual above 1000 MPa.

> Use `steels.loc[resid_all > 1000, [...]]`, with the column names in the list.

In [16]:
# your code here

(c) The test found in part (b) has a tensile strength of 6661 MPa.
Decide how to treat it before you refit.

- Say whether it is a recording error, a real extreme, or a different quantity from the rest.
- Say what evidence you would check, in one or two sentences in all.

> Use the other 914 tests as the reference for what a tensile strength in this table can be.

*your answer here*

(d) Refit by least squares without that test, to see how much it changes the fit.

- Make a mask `keep` that is `True` where `tensile` is below 2000 MPa.
- Print `np.sum(keep)`.
- Fit `proof[keep]` and `tensile[keep]` with `stats.linregress`, and store the result as `fit_keep`.
- Print the slope, rounded to three decimals.
- Print R², rounded to two decimals.

In [17]:
# your code here